In [2]:
import sqlite3
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from itertools import combinations
import re

#spark = SparkSession.builder.appName("FrequentItemsetMining").getOrCreate()

In [3]:
conn = sqlite3.connect('tiktok_breadth_first.db')
cursor = conn.cursor()

In [4]:
# Get all tables in the database
tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()

print("Tables in the database:")
for table in tables:
    print(f"\n{table[0]}:")
    # Get columns for each table
    columns = cursor.execute(f"PRAGMA table_info({table[0]});").fetchall()
    for col in columns:
        print(f"  - {col[1]} ({col[2]})")

Tables in the database:

users:
  - username (TEXT)
  - display_name (TEXT)
  - following_fetched (INTEGER)
  - reposts_fetched (INTEGER)

follow_relations:
  - from_username (TEXT)
  - to_username (TEXT)

videos:
  - reposter_username (TEXT)
  - id (TEXT)
  - create_time (INTEGER)
  - creator_username (TEXT)
  - video_description (TEXT)
  - hashtag_names (TEXT)
  - like_count (INTEGER)
  - comment_count (INTEGER)
  - view_count (INTEGER)
  - favorites_count (INTEGER)


In [13]:
query = """ 
SELECT hashtag_names 
FROM videos
JOIN follow_relations
ON follow_relations.from_username = videos.reposter_username
WHERE follow_relations.to_username = 'kamalahq'
"""

k_data = cursor.execute(query).fetchall()

OperationalError: no such table: videos

In [47]:
query = """ 
SELECT hashtag_names 
FROM videos
JOIN follow_relations
ON follow_relations.from_username = videos.reposter_username
WHERE follow_relations.to_username = 'teamtrump'
"""

t_data = cursor.execute(query).fetchall()

In [48]:
k_df = spark.createDataFrame(k_data, ['raw_hash'])
t_df = spark.createDataFrame(t_data, ['raw_hash'])

In [49]:
k_df.show(5)

+--------------------+
|            raw_hash|
+--------------------+
|                 fyp|
|blancaevangelista...|
|foryou,Sleep,bedt...|
|cardsagainsthuman...|
|                    |
+--------------------+
only showing top 5 rows



26/01/12 15:59:25 WARN TaskSetManager: Stage 87 contains a task of very large size (1580 KiB). The maximum recommended task size is 1000 KiB.


In [50]:
# Spark-native hashtag preprocessing: split by commas, remove non-alphanumerics, lowercase, drop 'fyp'
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Use existing DataFrame `k_df` and ensure `raw_hash` is a string without nulls
_df = k_df.withColumn("raw_hash", F.coalesce(F.col("raw_hash").cast("string"), F.lit("")))

# Remove any characters except letters, digits, and commas
_cleaned = F.regexp_replace(F.col("raw_hash"), r"[^a-zA-Z0-9,]+", "")
_tokens = F.split(_cleaned, ",")

# Lowercase and filter out empty strings and 'fyp'
_lowered = F.transform(_tokens, lambda x: F.lower(x))
_filtered = F.filter(_lowered, lambda x: (x != "") & (x != "fyp"))

# Add the cleaned list and a sequential id
w = Window.orderBy(F.monotonically_increasing_id())
k_df_clean = (
    _df
    .withColumn("clean_hash", _filtered)
    .withColumn("id", F.row_number().over(w))
)

_df = t_df.withColumn("raw_hash", F.coalesce(F.col("raw_hash").cast("string"), F.lit("")))

# Remove any characters except letters, digits, and commas
_cleaned = F.regexp_replace(F.col("raw_hash"), r"[^a-zA-Z0-9,]+", "")
_tokens = F.split(_cleaned, ",")

# Lowercase and filter out empty strings and 'fyp'
_lowered = F.transform(_tokens, lambda x: F.lower(x))
_filtered = F.filter(_lowered, lambda x: (x != "") & (x != "fyp"))

# Add the cleaned list and a sequential id
w = Window.orderBy(F.monotonically_increasing_id())
t_df_clean = (
    _df
    .withColumn("clean_hash", _filtered)
    .withColumn("id", F.row_number().over(w))
)

# Show sample
t_df_clean.select("id", "raw_hash", "clean_hash").show(5, truncate=False)

26/01/12 15:59:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+-------------------------------------------------------------+-------------------------------------------------------------------+
|id |raw_hash                                                     |clean_hash                                                         |
+---+-------------------------------------------------------------+-------------------------------------------------------------------+
|1  |overtime,fyp,trump2024                                       |[overtime, trump2024]                                              |
|2  |electrician,fyp,Viral                                        |[electrician, viral]                                               |
|3  |fy,construction,bluecollar,heat                              |[fy, construction, bluecollar, heat]                               |
|4  |fyp                                                          |[]                                                                 |
|5  |mgplumbing,TaylorSwift,dallastx,plumbersoft

In [51]:
k_df = k_df_clean.select('id', 'clean_hash')
t_df = t_df_clean.select('id', 'clean_hash')

In [52]:
def preprocess(x):
    """
    preprocessing strings with hashtags. commas are delimiters.
    need to filter out non-alphanumeric symbols

    IMPORTANT: filters out 'fyp' from list of hashtags
    """
    index = 0
    for row in x:
        index += 1
        hash_list = []
        rough_list = re.split(',', row['raw_hash'])
        for hash in rough_list:
            match_alphanum = re.findall(r'[a-zA-Z0-9]+', hash)
            for x in match_alphanum:
                if x.lower() != 'fyp':
                    hash_list.append(x.lower())
        row['id'] = index
        row['clean_hash'] = hash_list
        yield row

In [53]:
k_df.show(5)

26/01/12 15:59:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:26 WARN TaskSetManager: Stage 90 contains a task of very large size (1580 KiB). The maximum recommended task size is 1000 KiB.


+---+--------------------+
| id|          clean_hash|
+---+--------------------+
|  1|                  []|
|  2|[blancaevangelist...|
|  3|[foryou, sleep, b...|
|  4|[cardsagainsthuma...|
|  5|                  []|
+---+--------------------+
only showing top 5 rows



In [54]:
# support: minimum number s of posts an item needs to show up in
s_kamala = k_df.count() * 0.0025
L1 = (
    k_df.rdd.flatMap(lambda row: [((item,), 1) for item in row.clean_hash])
    .reduceByKey(lambda x, y: x + y)
    .filter(lambda x: x[1] >= s)
    .collect()
)

Lk_1_kamala = (dict(L1))
print("Frequent Singletons (L1)")

26/01/12 15:59:28 WARN TaskSetManager: Stage 92 contains a task of very large size (1580 KiB). The maximum recommended task size is 1000 KiB.
26/01/12 15:59:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:28 WARN TaskSetManager: Stage 95 contains a task of very large size (1580 KiB). The maximum recommended task size is 1000 KiB.
26/01/12 15:59:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:30 WARN WindowExec: No Partition Defined 

Frequent Singletons (L1)


In [55]:
s_trump = t_df.count() * 0.0025

L1 = (
    t_df.rdd.flatMap(lambda row: [((item,), 1) for item in row.clean_hash])
    .reduceByKey(lambda x, y: x + y)
    .filter(lambda x: x[1] >= s)
    .collect()
)

Lk_1_trump = (dict(L1))
print("Frequent Singletons (L1)")

26/01/12 15:59:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/01/12 15:59:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Frequent Singletons (L1)


In [56]:
Lk_1_kamala = {k: v for k, v in sorted(Lk_1_kamala.items(), key=lambda item: item[1], reverse=True)}
Lk_1_trump = {k: v for k, v in sorted(Lk_1_trump.items(), key=lambda item: item[1], reverse=True)}  

In [57]:
for item, count in Lk_1_trump.items():
    print(f"{item}: {count}")

('viral',): 16055
('foryou',): 7808
('foryoupage',): 7106
('edit',): 6509
('relatable',): 5503
('funny',): 4085
('trending',): 3065
('real',): 2759
('xyzbca',): 2516
('fy',): 2385
('fypviral',): 2350
('meme',): 1944
('blowthisup',): 1666
('capcut',): 1630
('viralvideo',): 1618
('fyppppppppppppppppppppppp',): 1590
('trend',): 1392
('fypage',): 1278
('fortnite',): 1276
('tiktok',): 1244
('ufc',): 1220
('football',): 1155
('goviral',): 1082
('love',): 1028
('dc',): 964
('nostalgia',): 961
('mma',): 890
('christmas',): 870
('blowup',): 809
('dexter',): 806
('school',): 794
('music',): 753
('motivation',): 738
('cat',): 722
('squidgame',): 710
('r6',): 692
('strangerthings',): 659
('clashroyale',): 645
('bro',): 617
('sad',): 614
('rainbowsixsiege',): 614
('dlaciebie',): 603
('roblox',): 591
('bigbang',): 574
('repost',): 565
('boxing',): 564
('relateable',): 554
('relationship',): 552
('gym',): 551
('xybca',): 546
('gaming',): 542
('usa',): 525
('jynxzi',): 520
('anime',): 514
('fypp',): 5

In [ ]:
for item, count in Lk_1_kamala.items():
    print(f"{item}: {count}")

('viral',): 72400
('foryoupage',): 44596
('foryou',): 43595
('relatable',): 29642
('edit',): 28175
('funny',): 19897
('billieeilish',): 18049
('xyzbca',): 17401
('trending',): 16524
('fypviral',): 13438
('fy',): 10952
('real',): 9465
('fyppppppppppppppppppppppp',): 9454
('trend',): 9081
('strangerthings',): 8548
('blowthisup',): 8471
('fypage',): 8249
('viralvideo',): 7799
('meme',): 7216
('billieeilishedits',): 6996
('roblox',): 6085
('school',): 5618
('capcut',): 5353
('music',): 5325
('tiktok',): 5240
('kpop',): 5204
('blowup',): 5172
('wlw',): 4588
('anime',): 4467
('goviral',): 4447
('aftereffects',): 4347
('xybca',): 3992
('strangerthings5',): 3991
('nostalgia',): 3858
('edits',): 3779
('love',): 3756
('billie',): 3656
('cat',): 3596
('lyrics',): 3361
('billieeilishedit',): 3319
('hmhas',): 3279
('fypp',): 3170
('christmas',): 3143
('xyzabc',): 3032
('aesthetic',): 2890
('catsoftiktok',): 2793
('fyppp',): 2792
('halloween',): 2790
('squidgame',): 2764
('sad',): 2676
('art',): 258

In [61]:
from pyspark.ml.fpm import FPGrowth

# FPGrowth requires the input column to be named 'items'

# Before running cell again: check to make sure all hashes are unique sets


fp_growth_k = FPGrowth(itemsCol="clean_hash", minSupport=(s_kamala / k_df.count()), minConfidence=0.8)
model_k = fp_growth_k.fit(k_df)

# Display frequent itemsets

#print("Frequent Itemsets found by FP-Growth:")
#model_k.freqItemsets.sort("freq", ascending=False).show()

# Display generated association rules
#print("Association Rules found by FP-Growth:")
#model_k.associationRules.sort("confidence", ascending=False).show()

26/01/12 16:03:53 WARN TaskSetManager: Stage 117 contains a task of very large size (1580 KiB). The maximum recommended task size is 1000 KiB.
26/01/12 16:03:54 WARN TaskSetManager: Stage 120 contains a task of very large size (1580 KiB). The maximum recommended task size is 1000 KiB.
26/01/12 16:03:56 WARN TaskSetManager: Stage 121 contains a task of very large size (1580 KiB). The maximum recommended task size is 1000 KiB.
26/01/12 16:03:56 WARN TaskSetManager: Stage 122 contains a task of very large size (1580 KiB). The maximum recommended task size is 1000 KiB.
26/01/12 16:03:56 ERROR Executor: Exception in task 14.0 in stage 122.0 (TID 733)
org.apache.spark.SparkException: Items in a transaction must be unique but got WrappedArray(baliindonesia, bali, switzerland, piramides, fypage, dream, stmoritz, indonesia, japanese, aesthetic, nyc, france, foryou, egypt, oldmoney, future, viral, motivation, greece, blowup, homedecor, grandprix, winter, brazil, world, paris, futurehouses, rome,

Py4JJavaError: An error occurred while calling o921.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 14 in stage 122.0 failed 1 times, most recent failure: Lost task 14.0 in stage 122.0 (TID 733) (midway3-0261.rcc.local executor driver): org.apache.spark.SparkException: Items in a transaction must be unique but got WrappedArray(baliindonesia, bali, switzerland, piramides, fypage, dream, stmoritz, indonesia, japanese, aesthetic, nyc, france, foryou, egypt, oldmoney, future, viral, motivation, greece, blowup, homedecor, grandprix, winter, brazil, world, paris, futurehouses, rome, house, mindset, monacolifestyle, fashion, foryoupage, popularity, millionare, houses, monacogp, brazil, fyppppppppppppppppppppppp, ny, success, f1, townhouse, fy, luxury, focus, life, aroundtheworld, japan, southafrica, villa, spain, home, middeternean, goviral, monaco, mncastr, luxurylife, italy, barcelona, billionare, penthouse, athens, sa, london, fypp).
	at org.apache.spark.mlli

Exception in thread "serve RDD 162" java.net.SocketTimeoutException: Accept timed out
	at java.base/sun.nio.ch.NioSocketImpl.timedAccept(NioSocketImpl.java:713)
	at java.base/sun.nio.ch.NioSocketImpl.accept(NioSocketImpl.java:757)
	at java.base/java.net.ServerSocket.implAccept(ServerSocket.java:675)
	at java.base/java.net.ServerSocket.platformImplAccept(ServerSocket.java:641)
	at java.base/java.net.ServerSocket.implAccept(ServerSocket.java:617)
	at java.base/java.net.ServerSocket.implAccept(ServerSocket.java:574)
	at java.base/java.net.ServerSocket.accept(ServerSocket.java:532)
	at org.apache.spark.security.SocketAuthServer$$anon$1.run(SocketAuthServer.scala:65)
26/01/12 16:11:03 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/blockmgr-f75eb0e7-d55e-42ec-9145-a547f97a71e6. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/blockmgr-f75eb0e7-d55e-42ec-9145-a547f97a71e6
	at org.apache.spark.network.util.JavaUtils.deleteRecursivel

In [ ]:
conn.close()

: 